show how to use QuantLib to generate schedules, i.e., regular sequences of dates, choosing from a number of market conventions. In turn, those can be used to create sequences of coupons; but that’s something for another time.

In [1]:
import QuantLib as ql
import pandas as pd

<h2>A simple example</h2>
We can build a schedule with as little information as a start date, an end date, and a frequency. Here is the corresponding call:

In [3]:
s = ql.MakeSchedule(
    effectiveDate=ql.Date(11, ql.May, 2021),
    terminationDate=ql.Date(11, ql.May, 2025),
    frequency=ql.Semiannual
)

pd.DataFrame(list(s), columns=['dates'])

,dates
0,"May 11th, 2021"
1,"November 11th, 2021"
2,"May 11th, 2022"
3,"November 11th, 2022"
4,"May 11th, 2023"
5,"November 11th, 2023"
6,"May 11th, 2024"
7,"November 11th, 2024"
8,"May 11th, 2025"


<h2>Adjusting for holidays</h2>
The above schedule didn’t make a distinction between holidays and business days. If we want holidays to be adjusted, we need to choose a calendar:

In [4]:
s = ql.MakeSchedule(
    effectiveDate=ql.Date(11, ql.May, 2021),
    terminationDate=ql.Date(11, ql.May, 2025),
    frequency=ql.Semiannual,
    calendar=ql.TARGET()
)

pd.DataFrame(list(s), columns=['dates'])

,dates
0,"May 11th, 2021"
1,"November 11th, 2021"
2,"May 11th, 2022"
3,"November 11th, 2022"
4,"May 11th, 2023"
5,"November 13th, 2023"
6,"May 13th, 2024"
7,"November 11th, 2024"
8,"May 12th, 2025"


As you can see, a few dates are no longer the 11th of the month and were replaced with the next business day. In this case, those dates fell on Saturdays or Sundays, but of course the adjustment would also be performed if they were mid-week holidays.

For convenience, it’s possible to pass a calendar and at the same time specify that dates should be unadjusted; here is the corresponding call:

In [5]:
s = ql.MakeSchedule(
    effectiveDate = ql.Date(11, ql.May, 2021),
    terminationDate = ql.Date(11, ql.May, 2025),
    frequency = ql.Semiannual,
    calendar = ql.TARGET(),
    convention=ql.Unadjusted
)

pd.DataFrame(list(s), columns=['Dates'])

,Dates
0,"May 11th, 2021"
1,"November 11th, 2021"
2,"May 11th, 2022"
3,"November 11th, 2022"
4,"May 11th, 2023"
5,"November 11th, 2023"
6,"May 11th, 2024"
7,"November 11th, 2024"
8,"May 11th, 2025"


As I said, this is a convenience; when reading data from a file or a DB, it makes it unnecessary to write logic that chooses whether or not to pass a calendar.

<h2>Short and long coupons</h2>
If the start and end dates don’t bracket a whole number of periods, it becomes important to specify whether the dates should be generated forwards from the start date or backwards from the end date; the default is to generate them backwards, but it’s probably better to be explicit. The corresponding calls are as follows:

In [8]:
s1 = ql.MakeSchedule(
    effectiveDate = ql.Date(11, ql.May, 2021),
    terminationDate= ql.Date(11, ql.February, 2025),
    frequency = ql.Semiannual,
    calendar= ql.TARGET(),
    forwards=True
)

s2 = ql.MakeSchedule(
    effectiveDate=ql.Date(11, ql.May, 2021),
    terminationDate=ql.Date(11, ql.February, 2025),
    frequency=ql.Semiannual,
    calendar=ql.TARGET(),
    backwards=True
)

pd.DataFrame(zip(s1, s2), columns=['forwards','backwards'])

,forwards,backwards
0,"May 11th, 2021","May 11th, 2021"
1,"November 11th, 2021","August 11th, 2021"
2,"May 11th, 2022","February 11th, 2022"
3,"November 11th, 2022","August 11th, 2022"
4,"May 11th, 2023","February 13th, 2023"
5,"November 13th, 2023","August 11th, 2023"
6,"May 13th, 2024","February 12th, 2024"
7,"November 11th, 2024","August 12th, 2024"
8,"February 11th, 2025","February 11th, 2025"


In the first case, we ended up with a short last coupon; in the second, with a short first coupon. It’s also possible to specify a long coupon by passing an explicit stub:

In [10]:
s=ql.MakeSchedule(
    effectiveDate=ql.Date(11, ql.February, 2021),
    terminationDate = ql.Date(11, ql.May, 2025),
    frequency = ql.Semiannual,
    firstDate=ql.Date(11, ql.November, 2021),
    calendar=ql.TARGET(),
    forwards=True
)

pd.DataFrame(list(s), columns=['dates'])

,dates
0,"February 11th, 2021"
1,"November 11th, 2021"
2,"May 11th, 2022"
3,"November 11th, 2022"
4,"May 11th, 2023"
5,"November 13th, 2023"
6,"May 13th, 2024"
7,"November 11th, 2024"
8,"May 12th, 2025"


In case of a long last coupon, you can use the nextToLastDate keyword instead of firstDate; the two can also be used together.

<h2>End of month</h2>

When the dates are close to the end of their month, other conventions can come into play. The default behavior is to generate dates as usual; in the call

In [13]:
s = ql.MakeSchedule(
    effectiveDate = ql.Date(28, ql.February, 2019),
    terminationDate=ql.Date(28, ql.February, 2023),
    frequency=ql.Semiannual,
    calendar=ql.TARGET(),
    forwards=True
)

pd.DataFrame(list(s), columns=['dates'])

,dates
0,"February 28th, 2019"
1,"August 28th, 2019"
2,"February 28th, 2020"
3,"August 28th, 2020"
4,"March 1st, 2021"
5,"August 30th, 2021"
6,"February 28th, 2022"
7,"August 29th, 2022"
8,"February 28th, 2023"


you can see that, for instance, February 28th 2021, a Sunday, is adjusted to March 1st according to the “following” convention. However, if another convention such as “modified following” needs to be used, it can be passed to the call:

In [14]:
s = ql.MakeSchedule(
    effectiveDate = ql.Date(28, ql.February, 2019),
    terminationDate=ql.Date(28, ql.February, 2023),
    frequency=ql.Semiannual,
    calendar=ql.TARGET(),
    convention=ql.ModifiedFollowing,
    forwards=True
)

pd.DataFrame(list(s), columns=['dates'])

,dates
0,"February 28th, 2019"
1,"August 28th, 2019"
2,"February 28th, 2020"
3,"August 28th, 2020"
4,"February 26th, 2021"
5,"August 30th, 2021"
6,"February 28th, 2022"
7,"August 29th, 2022"
8,"February 28th, 2023"


The result shows that February 28th 2021 is adjusted back to February 26th so that it doesn’t change month.

Also, in some cases, the terms of an instrument might stipulate that coupons reset at on the last business day of the month; in that case, the schedule can be generated with:

In [16]:
s = ql.MakeSchedule(
    effectiveDate=ql.Date(28, ql.February, 2019),
    terminationDate=ql.Date(28,ql.February, 2023),
    frequency=ql.Semiannual,
    calendar=ql.TARGET(),
    convention=ql.ModifiedFollowing,
    endOfMonth=True,
    forwards=True
)

pd.DataFrame(list(s), columns=['dates'])

,dates
0,"February 28th, 2019"
1,"August 30th, 2019"
2,"February 28th, 2020"
3,"August 31st, 2020"
4,"February 26th, 2021"
5,"August 31st, 2021"
6,"February 28th, 2022"
7,"August 31st, 2022"
8,"February 28th, 2023"


By comparing this result with the ones above, you can see the difference of behavior for the dates at the end of August.

<h2>Specialized Rules</h2>
The forwards and backwards methods shown above are shorthand for calls to a more general method withRule that allows to specify a generation rule (“forwards” and “backwards” being two such rules.) Other, more specialized rules are available; for instance, if you needed to generate the schedule for the payments of a standard 5-years credit default swap, you would do it as follows:

In [18]:
tradeDate = ql.Date(11, ql.March, 2021)
tenor = ql.Period(5, ql.Years)
maturityDate = ql.cdsMaturity(tradeDate, tenor, ql.DateGeneration.CDS2015)

In [20]:
s = ql.MakeSchedule(
    effectiveDate=tradeDate,
    terminationDate = maturityDate,
    frequency= ql.Quarterly,
    calendar=ql.TARGET(),
    rule=ql.DateGeneration.CDS2015
)

pd.DataFrame(list(s), columns=['dates'])

,dates
0,"December 21st, 2020"
1,"March 22nd, 2021"
2,"June 21st, 2021"
3,"September 20th, 2021"
4,"December 20th, 2021"
5,"March 21st, 2022"
6,"June 20th, 2022"
7,"September 20th, 2022"
8,"December 20th, 2022"
9,"March 20th, 2023"


First, the cdsMaturity function returns the standardized maturity date for the passed trade date; for March 11th 2021, that would be December 20th 2025 (it would roll to June 2025 only later in March.) Then, we pass the calculated maturity date to MakeSchedule while also specifying a CDS2015 date-generation rule; this recalculates the start date of the CDS and also adjusts all the dates in the schedule to the twentieth of their months of the next business day.